# DeBERTa 第五章实验 Notebook

本 Notebook 基于 `code/deberta_train_exp5.py` 重构，目标是直接产出第五章所需结果文件与可视化图。

输出目录默认：`outputs/deberta_ch5_experiments`


In [ ]:
import gc
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda count:', torch.cuda.device_count())


In [ ]:
# =========================
# 全局配置（Kaggle 双T4）
# =========================
ROOT = Path('.').resolve()
KAGGLE_INPUT = Path('/kaggle/input')

# 模型：优先你上传到input的本地目录，否则走HF
LOCAL_MODEL_CANDIDATES = [
    Path('/kaggle/input/deberta-v3-small'),
    Path('/kaggle/input/microsoft-deberta-v3-small'),
]
MODEL_CHECKPOINT = None
for m in LOCAL_MODEL_CANDIDATES:
    if m.exists():
        MODEL_CHECKPOINT = str(m)
        break
if MODEL_CHECKPOINT is None:
    MODEL_CHECKPOINT = 'microsoft/deberta-v3-small'


def find_first_exact(filename):
    if not KAGGLE_INPUT.exists():
        return None
    matches = list(KAGGLE_INPUT.rglob(filename))
    return matches[0] if matches else None


def find_first_contains(substr, suffix='.csv'):
    if not KAGGLE_INPUT.exists():
        return None
    substr = substr.lower()
    for p in KAGGLE_INPUT.rglob(f'*{suffix}'):
        if substr in p.name.lower():
            return p
    return None


# 关键文件按文件名自动发现（不依赖dataset目录名）
HUMAN_LLM_PARQUET = find_first_exact('data.parquet')
VALID_CSV = find_first_exact('nonTargetText_llm_slightly_modified_gen.csv')
if VALID_CSV is None:
    VALID_CSV = find_first_contains('nontargettext_llm_slightly_modified_gen', suffix='.csv')

PILE_FILES = {
    'pile2': find_first_exact('pile2.parquet'),
    'pile3': find_first_exact('plies3.parquet'),
    'pile4': find_first_exact('plies4.parquet'),
    'ultra': find_first_exact('Ultra.parquet'),
    'lmsys': find_first_exact('lmsys.parquet'),
}

# 输出写到 /kaggle/working
OUTPUT_DIR = Path('/kaggle/working/deberta_ch5_experiments')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_GROUPS = {
    'compare': True,
    'hparam': False,
    'multisource': False,
}

QUICK_MODE = True
EPOCHS_DEFAULT = 2 if QUICK_MODE else 4
EARLY_STOPPING_PATIENCE = 2 if QUICK_MODE else 4
MAX_LENGTH_DEFAULT = 256
LR_DEFAULT = 2e-5

# 双T4：batch_size 作为全局batch，训练函数内部会自动按卡数拆分
TARGET_NUM_GPUS = 2
AVAILABLE_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
EFFECTIVE_GPUS = min(TARGET_NUM_GPUS, AVAILABLE_GPUS) if AVAILABLE_GPUS > 0 else 0
BATCH_SIZE_DEFAULT = 8 if torch.cuda.is_available() else 8
GRAD_ACCUM_DEFAULT = 4 if torch.cuda.is_available() else 1

print('MODEL_CHECKPOINT =', MODEL_CHECKPOINT)
print('HUMAN_LLM_PARQUET =', HUMAN_LLM_PARQUET)
print('VALID_CSV =', VALID_CSV)
print('PILE_FILES =', {k: str(v) if v is not None else None for k, v in PILE_FILES.items()})
print('OUTPUT_DIR =', OUTPUT_DIR)
print(f'GPU available={AVAILABLE_GPUS}, target={TARGET_NUM_GPUS}, effective={EFFECTIVE_GPUS}')


In [ ]:
def softmax_np(logits):
    x = logits - np.max(logits, axis=-1, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=-1, keepdims=True)


def clean_text_series(s):
    return (
        s.fillna('')
         .astype(str)
         .str.replace(r'\r', ' ', regex=True)
         .str.replace(r'\n+', ' ', regex=True)
         .str.strip()
    )


def normalize_binary_label(x):
    if isinstance(x, str):
        xl = x.strip().lower()
        if xl in {'human', '0', 'false'}:
            return 0
        if xl in {'ai', '1', 'true'}:
            return 1
    try:
        return int(float(x))
    except Exception:
        return np.nan


def read_parquet_fallback(path):
    try:
        return pd.read_parquet(path, engine='fastparquet')
    except Exception:
        return pd.read_parquet(path, engine='pyarrow')


def prep_df(df, name):
    if 'text' not in df.columns:
        raise ValueError(f'{name} 缺少 text 列')

    if 'label' not in df.columns:
        if 'source' in df.columns:
            df = df.copy()
            df['label'] = np.where(df['source'].astype(str).str.lower() == 'human', 0, 1)
        else:
            raise ValueError(f'{name} 缺少 label/source 列，无法统一标签')

    out = df[['text', 'label']].copy()
    out['text'] = clean_text_series(out['text'])
    out['label'] = out['label'].apply(normalize_binary_label)
    out = out.dropna(subset=['text', 'label'])
    out['label'] = out['label'].astype(int)
    out = out[out['text'].str.len() > 0].reset_index(drop=True)
    return out


def balance_binary(df, seed=SEED):
    vc = df['label'].value_counts()
    if len(vc) < 2:
        return df
    max_n = vc.max()
    parts = []
    for y, n in vc.items():
        part = df[df['label'] == y]
        if n < max_n:
            part = part.sample(max_n, replace=True, random_state=seed)
        parts.append(part)
    out = pd.concat(parts, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out


In [ ]:
# =========================
# 读取基础数据源（内存安全版）
# =========================
source_frames = {}

# 每个源文件最多读取样本数（避免Kaggle内核重启）
SOURCE_SAMPLE_CAP = 60000 if QUICK_MODE else 120000


def read_prepare_sample(path, name, cap=SOURCE_SAMPLE_CAP):
    """按最小列读取 + 清洗 + 采样，降低内存压力"""
    # 不同数据源字段不同：优先 text/label，其次 text/source
    try:
        df = pd.read_parquet(path, columns=['text', 'label'], engine='pyarrow')
    except Exception:
        try:
            df = pd.read_parquet(path, columns=['text', 'source'], engine='pyarrow')
        except Exception:
            df = read_parquet_fallback(path)

    df = prep_df(df, name)
    if len(df) > cap:
        df = df.sample(cap, random_state=SEED).reset_index(drop=True)
    return df


# official: Human_LLM
if HUMAN_LLM_PARQUET is not None and Path(HUMAN_LLM_PARQUET).exists():
    source_frames['official'] = read_prepare_sample(HUMAN_LLM_PARQUET, 'official')
else:
    print('[WARN] official 数据不存在:', HUMAN_LLM_PARQUET)

# external A: pile2 + plies3
ext_a_parts = []
for key in ['pile2', 'pile3']:
    p = PILE_FILES.get(key)
    if p is not None and Path(p).exists():
        ext_a_parts.append(read_prepare_sample(p, str(Path(p).name)))
if ext_a_parts:
    external_A = pd.concat(ext_a_parts, axis=0).reset_index(drop=True)
    if len(external_A) > SOURCE_SAMPLE_CAP:
        external_A = external_A.sample(SOURCE_SAMPLE_CAP, random_state=SEED).reset_index(drop=True)
    source_frames['external_A'] = external_A
else:
    print('[WARN] external_A 数据未找到（pile2/plies3）')

# external B: plies4 + Ultra + lmsys
ext_b_parts = []
for key in ['pile4', 'ultra', 'lmsys']:
    p = PILE_FILES.get(key)
    if p is not None and Path(p).exists():
        ext_b_parts.append(read_prepare_sample(p, str(Path(p).name)))
if ext_b_parts:
    external_B = pd.concat(ext_b_parts, axis=0).reset_index(drop=True)
    if len(external_B) > SOURCE_SAMPLE_CAP:
        external_B = external_B.sample(SOURCE_SAMPLE_CAP, random_state=SEED).reset_index(drop=True)
    source_frames['external_B'] = external_B
else:
    print('[WARN] external_B 数据未找到（plies4/Ultra/lmsys）')

# 验证集：优先独立验证集；若缺失则从 official 切分
if VALID_CSV is not None and Path(VALID_CSV).exists():
    valid_official = prep_df(pd.read_csv(VALID_CSV), 'valid_csv')
    # 验证集太大也做上限（通常不会）
    if len(valid_official) > 80000:
        valid_official = valid_official.sample(80000, random_state=SEED).reset_index(drop=True)
    print('[INFO] 使用独立验证集:', VALID_CSV)
else:
    if 'official' not in source_frames:
        csv_candidates = sorted([str(x) for x in KAGGLE_INPUT.rglob('*.csv')]) if KAGGLE_INPUT.exists() else []
        print('[DEBUG] 可用CSV文件:')
        for x in csv_candidates[:50]:
            print(' -', x)
        raise FileNotFoundError('未找到 VALID_CSV，且 official 数据不可用，无法构建验证集。')

    from sklearn.model_selection import train_test_split
    _official_all = source_frames['official']
    tr, va = train_test_split(
        _official_all,
        test_size=0.2,
        random_state=SEED,
        stratify=_official_all['label'],
    )
    source_frames['official'] = tr.reset_index(drop=True)
    valid_official = va.reset_index(drop=True)
    print('[WARN] 未找到独立验证集，已从 official 数据按 8:2 自动切分验证集。')

for k, v in source_frames.items():
    print(f'{k}:', v.shape, 'label dist =', v['label'].value_counts().to_dict())
print('valid:', valid_official.shape, 'label dist =', valid_official['label'].value_counts().to_dict())

pd.DataFrame([
    {
        'dataset': k,
        'size': len(v),
        'pos_rate': round(v['label'].mean(), 6),
        'avg_len': round(v['text'].str.len().mean(), 2),
    }
    for k, v in source_frames.items()
]).to_csv(OUTPUT_DIR / 'dataset_source_stats.csv', index=False)


In [ ]:
def build_dataset_variant(
    name,
    use_official=True,
    use_external_A=True,
    use_external_B=True,
    dedup=True,
    balance=False,
):
    parts = []
    if use_official and 'official' in source_frames:
        parts.append(source_frames['official'])
    if use_external_A and 'external_A' in source_frames:
        parts.append(source_frames['external_A'])
    if use_external_B and 'external_B' in source_frames:
        parts.append(source_frames['external_B'])

    if not parts:
        raise ValueError(f'{name}: 没有可用训练源')

    train_df = pd.concat(parts, axis=0).reset_index(drop=True)

    if dedup:
        train_df = train_df.drop_duplicates(subset=['text']).reset_index(drop=True)

    if balance:
        train_df = balance_binary(train_df)


    # 防止Kaggle内存/显存崩溃：先限制训练集规模，跑通后再放开
    MAX_TRAIN_SAMPLES = 120000
    if len(train_df) > MAX_TRAIN_SAMPLES:
        train_df = train_df.sample(MAX_TRAIN_SAMPLES, random_state=SEED).reset_index(drop=True)

    train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    return {
        'name': name,
        'train_df': train_df,
        'valid_df': valid_official.copy(),
    }


In [ ]:
# =========================
# Tokenizer / Metrics / Callback
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=False)


def tokenize_function(examples, max_length=MAX_LENGTH_DEFAULT):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=max_length,
    )


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax_np(logits)[:, 1]
    preds = (probs >= 0.5).astype(int)

    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )
    roc = roc_auc_score(labels, probs)

    return {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'roc_auc': roc,
    }


class HistoryCallback(TrainerCallback):
    def __init__(self):
        self.logs = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            item = dict(logs)
            item['step'] = state.global_step
            item['epoch'] = state.epoch
            self.logs.append(item)


In [ ]:
def train_one_experiment(
    exp_name,
    train_df,
    valid_df,
    model_checkpoint=MODEL_CHECKPOINT,
    max_length=MAX_LENGTH_DEFAULT,
    learning_rate=LR_DEFAULT,
    batch_size=BATCH_SIZE_DEFAULT,
    grad_accum=GRAD_ACCUM_DEFAULT,
    num_epochs=EPOCHS_DEFAULT,
    weight_decay=0.01,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
):
    exp_dir = OUTPUT_DIR / exp_name
    exp_dir.mkdir(parents=True, exist_ok=True)

    # batch_size 视为全局batch，按双T4自动拆分到每卡
    visible_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
    effective_gpus = min(TARGET_NUM_GPUS, visible_gpus) if visible_gpus > 0 else 1
    per_device_batch = max(1, batch_size // effective_gpus)
    if batch_size % effective_gpus != 0:
        print(f'[WARN] global batch {batch_size} 不能整除 GPU {effective_gpus}，每卡batch改为 {per_device_batch}')

    ds_train = Dataset.from_pandas(train_df[['text', 'label']].reset_index(drop=True), preserve_index=False)
    ds_valid = Dataset.from_pandas(valid_df[['text', 'label']].reset_index(drop=True), preserve_index=False)

    ds_train = ds_train.map(lambda x: tokenize_function(x, max_length=max_length), batched=True)
    ds_valid = ds_valid.map(lambda x: tokenize_function(x, max_length=max_length), batched=True)

    if 'text' in ds_train.column_names:
        ds_train = ds_train.remove_columns(['text'])
    if 'text' in ds_valid.column_names:
        ds_valid = ds_valid.remove_columns(['text'])

    ds_train.set_format('torch')
    ds_valid.set_format('torch')

    model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

    history_cb = HistoryCallback()
    early_stopping = EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)

    train_args = TrainingArguments(
        output_dir=str(exp_dir / 'model'),
        eval_strategy='epoch',
        save_strategy='epoch',
        logging_strategy='steps',
        logging_steps=50,
        learning_rate=learning_rate,
        lr_scheduler_type='cosine',
        fp16=torch.cuda.is_available(),
        optim='adamw_torch',
        per_device_train_batch_size=per_device_batch,
        per_device_eval_batch_size=per_device_batch,
        gradient_accumulation_steps=grad_accum,
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
        ddp_find_unused_parameters=False if visible_gpus > 1 else None,
        num_train_epochs=num_epochs,
        weight_decay=weight_decay,
        load_best_model_at_end=True,
        metric_for_best_model='roc_auc',
        report_to='none',
        save_total_limit=2,
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=train_args,
        train_dataset=ds_train,
        eval_dataset=ds_valid,
        tokenizer=tokenizer,
        callbacks=[history_cb, early_stopping],
        compute_metrics=compute_metrics,
    )

    print('=' * 90)
    print('[START]', exp_name)
    print(f'train={len(train_df)}, valid={len(valid_df)}, max_len={max_length}, lr={learning_rate}, global_batch={batch_size}, per_device_batch={per_device_batch}, gpus={effective_gpus}, ga={grad_accum}, epochs={num_epochs}')

    trainer.train()

    pred_output = trainer.predict(ds_valid)
    logits = pred_output.predictions
    labels = pred_output.label_ids

    probs = softmax_np(logits)[:, 1]
    preds = (probs >= 0.5).astype(int)

    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    roc = roc_auc_score(labels, probs)

    result = {
        'exp_name': exp_name,
        'train_size': len(train_df),
        'valid_size': len(valid_df),
        'max_length': max_length,
        'learning_rate': learning_rate,
        'batch_size': batch_size,
        'per_device_batch': per_device_batch,
        'effective_gpus': effective_gpus,
        'grad_accum': grad_accum,
        'epochs': num_epochs,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'roc_auc': roc,
    }

    pd.DataFrame([result]).to_csv(exp_dir / 'metrics_summary.csv', index=False)
    pd.DataFrame({
        'text': valid_df['text'].values,
        'label': labels,
        'prob_ai': probs,
        'pred': preds,
    }).to_csv(exp_dir / 'valid_predictions.csv', index=False)

    history_df = pd.DataFrame(history_cb.logs)
    history_df.to_csv(exp_dir / 'train_history.csv', index=False)

    with open(exp_dir / 'config.json', 'w', encoding='utf-8') as f:
        json.dump({
            'model_checkpoint': model_checkpoint,
            'max_length': max_length,
            'learning_rate': learning_rate,
            'batch_size': batch_size,
            'per_device_batch': per_device_batch,
            'effective_gpus': effective_gpus,
            'grad_accum': grad_accum,
            'num_epochs': num_epochs,
            'weight_decay': weight_decay,
            'early_stopping_patience': early_stopping_patience,
        }, f, ensure_ascii=False, indent=2)

    if not history_df.empty:
        fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
        h = history_df.copy()

        if 'loss' in h.columns:
            hh = h.dropna(subset=['loss']).sort_values('step')
            if len(hh) > 0:
                ax[0].plot(hh['step'], hh['loss'], label='train_loss')

        if 'eval_loss' in h.columns:
            eh = h.dropna(subset=['eval_loss']).sort_values('step')
            if len(eh) > 0:
                ax[0].plot(eh['step'], eh['eval_loss'], label='eval_loss')

        ax[0].set_title('Loss Curve')
        ax[0].set_xlabel('Step')
        ax[0].set_ylabel('Loss')
        ax[0].grid(alpha=0.25)
        ax[0].legend()

        if 'eval_roc_auc' in h.columns:
            ah = h.dropna(subset=['eval_roc_auc']).sort_values('step')
            if len(ah) > 0:
                ax[1].plot(ah['step'], ah['eval_roc_auc'], marker='o', label='eval_roc_auc')

        ax[1].set_title('Validation AUC Curve')
        ax[1].set_xlabel('Step')
        ax[1].set_ylabel('AUC')
        ax[1].grid(alpha=0.25)
        ax[1].legend()

        fig.tight_layout()
        fig.savefig(exp_dir / 'loss_auc_curves.png', dpi=220)
        plt.close(fig)

    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result, history_df


In [ ]:
# =========================
# 表5-5：DeBERTa 主结果对比
# =========================
table55_rows = []

if RUN_GROUPS['compare']:
    compare_variants = [
        {
            'name': 'deberta_official_only',
            'dataset': build_dataset_variant('official_only', use_official=True, use_external_A=False, use_external_B=False, dedup=True, balance=False),
            'max_length': 384,
            'learning_rate': 2e-5,
            'batch_size': BATCH_SIZE_DEFAULT,
            'grad_accum': GRAD_ACCUM_DEFAULT,
            'epochs': EPOCHS_DEFAULT,
        },
        {
            'name': 'deberta_official_externalA',
            'dataset': build_dataset_variant('official_externalA', use_official=True, use_external_A=True, use_external_B=False, dedup=True, balance=False),
            'max_length': 384,
            'learning_rate': 2e-5,
            'batch_size': BATCH_SIZE_DEFAULT,
            'grad_accum': GRAD_ACCUM_DEFAULT,
            'epochs': EPOCHS_DEFAULT,
        },
        {
            'name': 'deberta_official_externalAB',
            'dataset': build_dataset_variant('official_externalAB', use_official=True, use_external_A=True, use_external_B=True, dedup=True, balance=False),
            'max_length': 384,
            'learning_rate': 2e-5,
            'batch_size': BATCH_SIZE_DEFAULT,
            'grad_accum': GRAD_ACCUM_DEFAULT,
            'epochs': EPOCHS_DEFAULT,
        },
    ]

    for cfg in compare_variants:
        res, hist = train_one_experiment(
            exp_name=cfg['name'],
            train_df=cfg['dataset']['train_df'],
            valid_df=cfg['dataset']['valid_df'],
            max_length=cfg['max_length'],
            learning_rate=cfg['learning_rate'],
            batch_size=cfg['batch_size'],
            grad_accum=cfg['grad_accum'],
            num_epochs=cfg['epochs'],
        )

        table55_rows.append({
            '模型': 'DeBERTa-v3-small',
            '训练数据': cfg['dataset']['name'],
            'max_length': cfg['max_length'],
            'Accuracy': round(res['accuracy'], 6),
            'Precision': round(res['precision'], 6),
            'Recall': round(res['recall'], 6),
            'F1-score': round(res['f1'], 6),
            'ROC-AUC': round(res['roc_auc'], 6),
        })

    table55 = pd.DataFrame(table55_rows)
    table55.to_csv(OUTPUT_DIR / 'table_5_5_deberta_main.csv', index=False)
    display(table55)


In [ ]:
# =========================
# 表5-8：超参数实验
# =========================
hparam_rows = []

if RUN_GROUPS['hparam']:
    base_ds = build_dataset_variant('official_externalAB_for_hparam', use_official=True, use_external_A=True, use_external_B=True, dedup=True, balance=False)

    hparam_grid = [
        {'exp_id': 'E1', 'lr': 2e-5, 'batch': BATCH_SIZE_DEFAULT, 'grad_accum': GRAD_ACCUM_DEFAULT, 'epochs': 3, 'max_length': 256},
        {'exp_id': 'E2', 'lr': 2e-5, 'batch': BATCH_SIZE_DEFAULT, 'grad_accum': GRAD_ACCUM_DEFAULT, 'epochs': 5 if not QUICK_MODE else 3, 'max_length': 256},
        {'exp_id': 'E3', 'lr': 3e-5, 'batch': max(2, BATCH_SIZE_DEFAULT // 2), 'grad_accum': GRAD_ACCUM_DEFAULT + 1, 'epochs': 3, 'max_length': 384},
        {'exp_id': 'E4', 'lr': 5e-5, 'batch': max(2, BATCH_SIZE_DEFAULT // 2), 'grad_accum': GRAD_ACCUM_DEFAULT + 1, 'epochs': 3, 'max_length': 384},
    ]

    for cfg in hparam_grid:
        exp_name = f'deberta_hparam_{cfg["exp_id"]}'
        res, hist = train_one_experiment(
            exp_name=exp_name,
            train_df=base_ds['train_df'],
            valid_df=base_ds['valid_df'],
            max_length=cfg['max_length'],
            learning_rate=cfg['lr'],
            batch_size=cfg['batch'],
            grad_accum=cfg['grad_accum'],
            num_epochs=cfg['epochs'],
        )

        hparam_rows.append({
            '实验编号': cfg['exp_id'],
            'learning_rate': cfg['lr'],
            'batch_size': cfg['batch'],
            'grad_accum': cfg['grad_accum'],
            'epoch': cfg['epochs'],
            'max_length': cfg['max_length'],
            'early_stopping': '是',
            'ROC-AUC': round(res['roc_auc'], 6),
            'Accuracy': round(res['accuracy'], 6),
            'Precision': round(res['precision'], 6),
            'Recall': round(res['recall'], 6),
            'F1-score': round(res['f1'], 6),
        })

    table58 = pd.DataFrame(hparam_rows)
    table58.to_csv(OUTPUT_DIR / 'table_5_8_deberta_hparams.csv', index=False)
    display(table58)


In [ ]:
# =========================
# 表5-9：多源数据融合实验
# =========================
multi_rows = []

if RUN_GROUPS['multisource']:
    variants = [
        {'name': 'S1', 'cfg': dict(use_official=True, use_external_A=False, use_external_B=False, dedup=True, balance=False)},
        {'name': 'S2', 'cfg': dict(use_official=True, use_external_A=True, use_external_B=False, dedup=True, balance=False)},
        {'name': 'S3', 'cfg': dict(use_official=True, use_external_A=True, use_external_B=True, dedup=True, balance=False)},
        {'name': 'S4', 'cfg': dict(use_official=True, use_external_A=True, use_external_B=True, dedup=True, balance=True)},
    ]

    for item in variants:
        ds = build_dataset_variant(name=item['name'], **item['cfg'])
        exp_name = f'deberta_multisource_{item["name"]}'

        res, hist = train_one_experiment(
            exp_name=exp_name,
            train_df=ds['train_df'],
            valid_df=ds['valid_df'],
            max_length=384,
            learning_rate=2e-5,
            batch_size=max(2, BATCH_SIZE_DEFAULT // 2),
            grad_accum=max(1, GRAD_ACCUM_DEFAULT + 1),
            num_epochs=EPOCHS_DEFAULT,
        )

        multi_rows.append({
            '训练数据方案': item['name'],
            '官方数据': '√' if item['cfg']['use_official'] else '×',
            '外部数据A': '√' if item['cfg']['use_external_A'] else '×',
            '外部数据B': '√' if item['cfg']['use_external_B'] else '×',
            '去重': '√' if item['cfg']['dedup'] else '×',
            '平衡采样': '√' if item['cfg']['balance'] else '×',
            'ROC-AUC': round(res['roc_auc'], 6),
            'Accuracy': round(res['accuracy'], 6),
            'Precision': round(res['precision'], 6),
            'Recall': round(res['recall'], 6),
            'F1-score': round(res['f1'], 6),
        })

    table59 = pd.DataFrame(multi_rows)
    table59.to_csv(OUTPUT_DIR / 'table_5_9_deberta_multisource.csv', index=False)
    display(table59)


In [ ]:
# =========================
# 汇总与最佳实验
# =========================
metric_files = list(OUTPUT_DIR.glob('*/metrics_summary.csv'))
if metric_files:
    all_metrics = pd.concat([pd.read_csv(f) for f in metric_files], axis=0).reset_index(drop=True)
    all_metrics = all_metrics.sort_values('roc_auc', ascending=False).reset_index(drop=True)
    all_metrics.to_csv(OUTPUT_DIR / 'all_experiment_metrics.csv', index=False)
    display(all_metrics.head(10))

    best = all_metrics.iloc[0].to_dict()
    with open(OUTPUT_DIR / 'final_summary.json', 'w', encoding='utf-8') as f:
        json.dump({
            'best_experiment': best,
            'all_metric_files': [str(x) for x in metric_files],
            'output_dir': str(OUTPUT_DIR),
        }, f, ensure_ascii=False, indent=2)

print('Done. outputs at:', OUTPUT_DIR)
